<a href="https://colab.research.google.com/github/jshingala/FlashAttention/blob/Tri/trition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Int

In [ ]:
import torch
import triton

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"Triton version: {triton.__version__}")

CUDA available: True
Device: Tesla T4
Triton version: 3.6.0


In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def copy_kernel(
    x_ptr,            # pointer to input tensor in GPU memory
    out_ptr,          # pointer to output tensor in GPU memory
    n_elements,       # total number of elements
    BLOCK_SIZE: tl.constexpr,   # how many elements each program handles
):
    # 1. Which block am I? Programs are numbered 0, 1, 2, ...
    pid = tl.program_id(axis=0)

    # 2. Compute the range of indices THIS program is responsible for.
    #    If BLOCK_SIZE=1024 and pid=2, this covers indices 2048..3071.
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)

    # 3. Mask: the last block may run past the end of the tensor.
    #    This says "only touch indices that actually exist."
    mask = offsets < n_elements

    # 4. Load my block from global memory (masked), compute, store it back.
    x = tl.load(x_ptr + offsets, mask=mask)
    tl.store(out_ptr + offsets, x, mask=mask)


def copy(x: torch.Tensor):
    out = torch.empty_like(x)
    n_elements = x.numel()
    BLOCK_SIZE = 1024
    # The "grid" = how many programs to launch. We need enough blocks
    # to cover all elements: ceil(n_elements / BLOCK_SIZE).
    grid = (triton.cdiv(n_elements, BLOCK_SIZE),)
    copy_kernel[grid](x, out, n_elements, BLOCK_SIZE=BLOCK_SIZE)
    return out


# --- test it ---
x = torch.randn(10000, device="cuda")
out = copy(x)
print("Max difference from input:", (out - x).abs().max().item())
print("Match:", torch.allclose(out, x))

Max difference from input: 0.0
Match: True


# Stage 2

In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def softmax_kernel(
    x_ptr,              # input  [n_rows, n_cols]
    out_ptr,            # output [n_rows, n_cols]
    row_stride,         # how many elements to skip to get to the next row
    n_cols,             # number of columns (real width of each row)
    BLOCK_SIZE: tl.constexpr,   # padded width (power of two >= n_cols)
):
    # 1. Which ROW am I responsible for?
    row_idx = tl.program_id(axis=0)

    # 2. Column offsets + pointers for this row.
    col_offsets = tl.arange(0, BLOCK_SIZE)
    ptrs = x_ptr + row_idx * row_stride + col_offsets

    # 3. Mask out the padding columns (BLOCK_SIZE may exceed n_cols).
    mask = col_offsets < n_cols

    # 4. Load the row; padding slots load -inf so exp(-inf)=0.
    row = tl.load(ptrs, mask=mask, other=float("-inf"))

    # 5. Stable softmax.
    row_minus_max = row - tl.max(row, axis=0)
    numerator = tl.exp(row_minus_max)
    denominator = tl.sum(numerator, axis=0)
    softmax_out = numerator / denominator

    # 6. Store back to the same positions.
    out_ptrs = out_ptr + row_idx * row_stride + col_offsets
    tl.store(out_ptrs, softmax_out, mask=mask)


def softmax(x: torch.Tensor):
    n_rows, n_cols = x.shape
    # BLOCK_SIZE must be a power of two >= n_cols
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    out = torch.empty_like(x)
    # One program per row → grid is (n_rows,)
    grid = (n_rows,)
    softmax_kernel[grid](
        x, out,
        x.stride(0),     # row stride
        n_cols,
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return out


# --- test against PyTorch ---
x = torch.randn(1823, 781, device="cuda")   # deliberately not round numbers
out_triton = softmax(x)
out_torch = torch.softmax(x, dim=-1)
print("Max difference:", (out_triton - out_torch).abs().max().item())
print("Match:", torch.allclose(out_triton, out_torch, atol=1e-5))

Max difference: 7.450580596923828e-09
Match: True


# Stage 3

In [1]:
import torch
import triton
import triton.language as tl


@triton.jit
def flash_attention_kernel(
    q_ptr, k_ptr, v_ptr, out_ptr,
    stride_qh, stride_qm, stride_qd,
    stride_kh, stride_kn, stride_kd,
    stride_vh, stride_vn, stride_vd,
    stride_oh, stride_om, stride_od,
    N,                              # sequence length
    scale,                          # 1 / sqrt(head_dim)
    BLOCK_M: tl.constexpr,          # query rows per program
    BLOCK_N: tl.constexpr,          # key/value rows per inner step
    HEAD_DIM: tl.constexpr,         # head dimension D
):
    # Which query block, and which (batch*head) am I?  [GIVEN]
    pid_m = tl.program_id(axis=0)
    pid_h = tl.program_id(axis=1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)   # query rows this program owns
    offs_d = tl.arange(0, HEAD_DIM)                     # the full head dimension

    # Load this program's Q tile once: [BLOCK_M, HEAD_DIM]  [GIVEN]
    q_ptrs = q_ptr + pid_h * stride_qh \
        + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qd
    q_mask = offs_m[:, None] < N
    q = tl.load(q_ptrs, mask=q_mask, other=0.0)

    # ---- TODO 1: initialize the running state ----
    m_i = tl.full((BLOCK_M,), float("-inf"), dtype=tl.float32)   # running max
    l_i = tl.zeros((BLOCK_M,), dtype=tl.float32)                 # running sum of exps
    acc = tl.zeros((BLOCK_M, HEAD_DIM), dtype=tl.float32)        # running weighted output

    # Walk the keys/values in blocks of BLOCK_N.  [loop GIVEN; K/V loads GIVEN]
    for start_n in range(0, N, BLOCK_N):
        offs_n = start_n + tl.arange(0, BLOCK_N)

        k_ptrs = k_ptr + pid_h * stride_kh \
            + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
        v_ptrs = v_ptr + pid_h * stride_vh \
            + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd
        kv_mask = offs_n[:, None] < N
        k = tl.load(k_ptrs, mask=kv_mask, other=0.0)   # [BLOCK_N, HEAD_DIM]
        v = tl.load(v_ptrs, mask=kv_mask, other=0.0)   # [BLOCK_N, HEAD_DIM]

        # ---- TODO 2: scores for this block ----
        s = tl.dot(q, tl.trans(k)) * scale                       # [BLOCK_M, BLOCK_N]
        s = tl.where(offs_n[None, :] < N, s, float("-inf"))      # kill out-of-range keys

        # ---- TODO 3: the online-softmax recurrence ----
        m_new = tl.maximum(m_i, tl.max(s, axis=1))               # new running max
        alpha = tl.exp(m_i - m_new)                              # re-base old state
        p     = tl.exp(s - m_new[:, None])                       # stable block weights
        l_i   = l_i * alpha + tl.sum(p, axis=1)                  # rescale sum, add block
        acc   = acc * alpha[:, None] + tl.dot(p.to(v.dtype), v)  # rescale out, add block
        m_i   = m_new                                            # carry max forward

    # ---- TODO 4: normalize exactly ONCE, after the loop ----
    acc = acc / l_i[:, None]

    # Store the [BLOCK_M, HEAD_DIM] output tile.  [GIVEN]
    out_ptrs = out_ptr + pid_h * stride_oh \
        + offs_m[:, None] * stride_om + offs_d[None, :] * stride_od
    tl.store(out_ptrs, acc, mask=q_mask)


def flash_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor):
    # q, k, v: [B, H, N, D]  — host side is GIVEN, it's the same launch
    # mechanics you already did. Focus on the kernel.
    B, H, N, D = q.shape
    q = q.reshape(B * H, N, D)
    k = k.reshape(B * H, N, D)
    v = v.reshape(B * H, N, D)
    out = torch.empty_like(q)

    scale = 1.0 / (D ** 0.5)
    BLOCK_M, BLOCK_N = 64, 64
    grid = (triton.cdiv(N, BLOCK_M), B * H)

    flash_attention_kernel[grid](
        q, k, v, out,
        q.stride(0), q.stride(1), q.stride(2),
        k.stride(0), k.stride(1), k.stride(2),
        v.stride(0), v.stride(1), v.stride(2),
        out.stride(0), out.stride(1), out.stride(2),
        N, scale,
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, HEAD_DIM=D,
    )
    return out.reshape(B, H, N, D)


# --- test against PyTorch (GIVEN) ---
torch.manual_seed(0)
B, H, N, D = 2, 4, 512, 64
q = torch.randn(B, H, N, D, device="cuda")
k = torch.randn(B, H, N, D, device="cuda")
v = torch.randn(B, H, N, D, device="cuda")

out_triton = flash_attention(q, k, v)
scores = (q @ k.transpose(-2, -1)) / (D ** 0.5)
out_torch = torch.softmax(scores, dim=-1) @ v

print("Max difference:", (out_triton - out_torch).abs().max().item())
print("Match:", torch.allclose(out_triton, out_torch, atol=1e-2))

Max difference: 4.76837158203125e-07
Match: True


# Stage 4


In [5]:
import torch
import triton

# Assumes flash_attention(q, k, v) is defined (Stage 3).
# Stage 4's headline is the MEMORY curve: naive attention's peak memory grows
# O(N^2) (it materializes the full NxN scores matrix) while Flash stays O(N).
# Throughput is the secondary table. This harness measures both, and lets
# naive OOM loudly at the size where that divergence becomes the whole point.


def time_ms(fn, *args, warmup=25, iters=100):
    """Median GPU time in ms. CUDA events (launches are async, so a CPU timer
    measures launch overhead, not compute); warmup excluded; median over iters."""
    for _ in range(warmup):
        fn(*args)
    torch.cuda.synchronize()
    ts = []
    for _ in range(iters):
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        fn(*args)
        e.record()
        e.synchronize()
        ts.append(s.elapsed_time(e))
    ts.sort()
    return ts[len(ts) // 2]


def peak_mem_mb(fn, *args):
    """Peak CUDA memory (MB) attributable to one call of fn.
    reset_peak_memory_stats zeroes the high-water mark; we run once; then read
    max_memory_allocated. Returns ('OOM', None) if the call runs out of memory."""
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()
    try:
        out = fn(*args)
        torch.cuda.synchronize()
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return None
    peak = torch.cuda.max_memory_allocated()
    del out
    return (peak - base) / 1e6


def naive_attention(q, k, v):
    scale = 1.0 / (q.shape[-1] ** 0.5)
    scores = (q @ k.transpose(-2, -1)) * scale   # [B,H,N,N] — the O(N^2) tensor
    return torch.softmax(scores, dim=-1) @ v


def sdpa(q, k, v):
    return torch.nn.functional.scaled_dot_product_attention(q, k, v)


def flops_attention(B, H, N, D):
    return 4.0 * B * H * N * N * D   # 2 matmuls, 2*N*N*D MACs each, 2 FLOP/MAC


def run(dtype=torch.float16):
    if not torch.cuda.is_available():
        raise SystemExit("No CUDA device. This harness must run on a GPU.")

    # NOTE on dtype: default fp16. SDPA's flash backend usually won't dispatch
    # at fp32, so the honest "vs SDPA" comparison is fp16. Your kernel keeps an
    # fp32 accumulator internally (the p.to(v.dtype) cast), so fp16 inputs are
    # the intended production path. Pass dtype=torch.float32 to see the gap.
    torch.manual_seed(0)
    configs = [
        (1, 16, 512,  64),
        (1, 16, 1024, 64),
        (1, 16, 2048, 64),
        (1, 16, 4096, 64),
        (1, 16, 8192, 64),
        (1, 16, 16384, 64),   # naive is expected to OOM somewhere in here
    ]

    print(f"dtype={dtype}")
    print(f"{'N':>7} | {'corr':>5} | "
          f"{'naive MB':>10} | {'flash MB':>10} | {'sdpa MB':>10} | "
          f"{'naive ms':>9} | {'flash ms':>9} | {'sdpa ms':>9} | {'flash TFLOP/s':>13}")
    print("-" * 110)

    for (B, H, N, D) in configs:
        q = torch.randn(B, H, N, D, device="cuda", dtype=dtype)
        k = torch.randn(B, H, N, D, device="cuda", dtype=dtype)
        v = torch.randn(B, H, N, D, device="cuda", dtype=dtype)

        # correctness gate (only meaningful while naive still fits)
        try:
            out_t = flash_attention(q, k, v)
            out_ref = naive_attention(q, k, v)
            ok = torch.allclose(out_t, out_ref, atol=1e-2 if dtype == torch.float32 else 2e-2)
            corr = "ok" if ok else "FAIL"
            del out_ref
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            corr = "n/a"   # naive OOM'd — can't compare, and that's the story

        # memory
        m_naive = peak_mem_mb(naive_attention, q, k, v)
        m_flash = peak_mem_mb(flash_attention, q, k, v)
        m_sdpa = peak_mem_mb(sdpa, q, k, v)

        # timing — skip naive if it OOM'd on memory
        t_naive = "OOM" if m_naive is None else f"{time_ms(naive_attention, q, k, v):.3f}"
        t_flash = time_ms(flash_attention, q, k, v)
        t_sdpa = time_ms(sdpa, q, k, v)

        tflops_flash = flops_attention(B, H, N, D) / (t_flash * 1e-3) / 1e12

        def fmt_mb(m):
            return "OOM" if m is None else f"{m:.1f}"

        print(f"{N:>7} | {corr:>5} | "
              f"{fmt_mb(m_naive):>10} | {fmt_mb(m_flash):>10} | {fmt_mb(m_sdpa):>10} | "
              f"{t_naive:>9} | {t_flash:>9.3f} | {t_sdpa:>9.3f} | {tflops_flash:>13.1f}")

        del q, k, v
        torch.cuda.empty_cache()


if __name__ == "__main__":
    run(dtype=torch.float16)   # run(dtype=torch.float32) for the apples-to-apples gap

dtype=torch.float16
      N |  corr |   naive MB |   flash MB |    sdpa MB |  naive ms |  flash ms |   sdpa ms | flash TFLOP/s
--------------------------------------------------------------------------------------------------------------
    512 |    ok |       17.8 |        1.0 |        1.0 |     0.385 |     2.376 |     0.137 |           0.5
   1024 |    ok |       69.2 |        2.1 |        2.1 |     1.086 |     8.747 |     0.400 |           0.5
   2048 |    ok |      272.6 |        4.2 |        4.2 |     4.342 |    34.778 |     1.547 |           0.5
   4096 |    ok |     1082.1 |        8.4 |        8.4 |    20.085 |   137.969 |     5.855 |           0.5
   8192 |    ok |     4311.7 |       16.8 |       16.8 |    77.542 |   547.192 |    22.204 |           0.5


KeyboardInterrupt: 